In [30]:
#Step 1

import pandas as pd
df_insurance = pd.read_csv('auto_insurance_python.csv')
df_insurance = df_insurance.dropna(subset=['State'])
df_insurance

,Customer ID,State,Coverage,Vehicle Class,Income,Months Since Policy Inception,Number of Policies,Customer Lifetime Revenue,Response
0,AA71604,Arizona,Extended,Two-Door Car,87560,86,2,8428,0.0
1,AB13432,California,Basic,SUV,0,16,1,1760,1.0
2,AB21519,California,Extended,Four-Door Car,93272,54,1,5130,0.0
3,AB31813,Washington,Extended,SUV,0,59,2,7729,0.0
4,AB60627,California,Extended,SUV,77517,20,2,2580,1.0
...,...,...,...,...,...,...,...,...,...
1995,NV14535,Nevada,Basic,SUV,20228,43,2,4343,NaN
1996,PV55726,Oregon,Basic,SUV,29462,23,1,2461,NaN
1997,QI40455,Oregon,Basic,SUV,0,79,2,8453,NaN
1998,UO86707,Washington,Basic,Four-Door Car,75217,47,2,2867,NaN


In [31]:
#Question 1.1

df_insurance['Income'].mean()

np.float64(39005.91032745592)

In [32]:
#Question 1.2

df_insurance.groupby('State')['Number of Policies'].mean()[['Arizona','Oregon']]

State
Arizona    3.200514
Oregon     2.880200
Name: Number of Policies, dtype: float64

In [33]:
#Question 1.3

(df_insurance['Income']==0).mean()*100
#A reason why some people might be reporting an income of 0 could be because the person who holds the policy could have no earned income. 
#They may be a student or someone who is retired. Instead of just keeping these as zeros, we could standarize the incomes that are not 0 so that the incomes that are a lot larger may not pose as an advantage over numbers that are zero or closer to zero.
#But even in a predictive model like a decision tree, we could keep the incomes as zero and use them as a separate branch and split on income that is greater than zero or income that is equal to 0. 

np.float64(23.828715365239294)

In [34]:
#Step 2.1

import numpy as np

binning = [df_insurance['Number of Policies']==1,(df_insurance['Number of Policies']>1)&(df_insurance['Number of Policies']<=5),df_insurance['Number of Policies']>5,]
choices = [1,2,3]
df_insurance['Number of Policies'] = np.select(binning,choices)

In [35]:
#Step 2.2

df_insurance = df_insurance.drop(columns=['Customer ID'])

In [36]:
#Step 2.3
df_insurance = pd.get_dummies(df_insurance,columns=['State','Coverage','Vehicle Class'],drop_first=True,dtype=int)

In [37]:
#Step 2.4

from sklearn.preprocessing import StandardScaler

columns = ['Income','Months Since Policy Inception','Customer Lifetime Revenue']

scaler = StandardScaler()
df_insurance[columns] = scaler.fit_transform(df_insurance[columns])
df_insurance[columns]

,Income,Months Since Policy Inception,Customer Lifetime Revenue
0,1.588512,1.356577,1.173531
1,-1.276131,-1.154699,-0.836038
2,1.775388,0.208565,0.179596
3,-1.276131,0.387942,0.962869
4,1.259942,-1.011197,-0.588911
...,...,...,...
1995,-0.614345,-0.186064,-0.057587
1996,-0.312242,-0.903571,-0.624774
1997,-1.276131,1.105449,1.181065
1998,1.184695,-0.042562,-0.502416


In [38]:
#Question 2.1
df_insurance['Number of Policies'].value_counts().sort_index()

Number of Policies
1    718
2    914
3    353
Name: count, dtype: int64

In [39]:
#Question 2.2

df_insurance['Income'].describe()
#max income is 1.994

count    1.985000e+03
mean     8.948901e-17
std      1.000252e+00
min     -1.276131e+00
25%     -8.548752e-01
50%     -1.146024e-01
75%      8.218048e-01
max      1.994882e+00
Name: Income, dtype: float64

In [40]:
#Question 2.3

number_attributes = len(df_insurance.columns)
print(number_attributes)

16


In [41]:
#Question 2.4

#No because the binned attributes of the Number of Policies are categorical ordinal variables.
#The intervals between them are not equal (first bin is only 1, while second bin 2-5 is 4). Using sample mean and standard deviation wouldn't make sense since the distance between these bins are not equal.

In [42]:
#Step 3.1

df_prediction = df_insurance[df_insurance['Response'].isna()].copy()

In [43]:
#Step 3.2

df_insurance = df_insurance.dropna(subset=['Response'])

In [44]:
#Question 3

print("Observations in df_prediction:", len(df_prediction))
print("Observations in df_insurance:", len(df_insurance))

Observations in df_prediction: 25
Observations in df_insurance: 1960


In [45]:
#Step 4.1

X = df_insurance.drop(columns=['Response'])
y = df_insurance['Response']

In [47]:
#Step 4.2

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

In [48]:
#Step 4.3
from sklearn.neural_network import MLPClassifier
nn_model = MLPClassifier(hidden_layer_sizes=(5,), random_state=1)
nn_model.fit(X_train, y_train)

,hidden_layer_sizes,"(5,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,1


In [49]:
#Step 4.4
y_pred_test = nn_model.predict(X_test)

In [ ]:
#Question 4.1

#No, standardizing the attributes before splitting them isn't an appropriate step. Since the test set is supposed to simulate data that is unseen so that it can give an unbiased prediction of the model's generalization, inclduing the test samples when we calculated mean and standard deviation will enable the model to train on scales 
#that are influenced by the data that it was already exposed to when it tested it, leading to more accurate scores.
#So to correct this, we must split before any feature transformations can learn parameters like mean or standard deviation

In [57]:
#Question 4.2

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, pos_label=1.0)
recall = recall_score(y_test, y_pred_test, pos_label=1.0)

print('Accuracy:', round(accuracy,4))
print('Precision:', round(precision,4))
print('Recall:', round(recall,4))

print("\nConfusion Matrix:", confusion_matrix(y_test, y_pred_test))

Accuracy: 0.5136
Precision: 0.5142
Recall: 0.67

Confusion Matrix: [[103 188]
 [ 98 199]]


In [ ]:
#Question 4.3

#The model is making more false positve errors on the test data s et. 
#From the confusion matrix, we can see that it made 188 false positives while it made 98 false negatives.
# Since this is the case, we would want to maximize precision since it would minimize the proportion of false positives throughout all of the outreach calls.
#We would then see that marketing resources are distributed to the customers with the best chance of renewing the policy.

In [64]:
#Step 5
X_prediction = df_prediction.drop(columns=['Response'])
df_prediction['Predicted_Response'] = nn_model.predict(X_prediction).astype(int)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Predicted_Response


In [62]:
#Question 5
accept_count = (df_prediction['Predicted_Response'] == 1).sum()
reject_count = (df_prediction['Predicted_Response'] == 0).sum()

print('Predicted to Accept:',accept_count)
print('Predicted to Reject:',reject_count)

Predicted to Accept: 18
Predicted to Reject: 7


In [ ]:
#Step 6.1
X_prediction = df_prediction.drop(columns=['Response'])
df_prediction['Prob_Class_1'] = nn_model.predict_proba(X_prediction)[:, 1]


In [69]:
#Step 6.2
predicted_class_06 = (df_prediction['Prob_Class_1'] >= 0.60).astype(int)
count_class_1 = (predicted_class_06 == 1).sum()
print("Number of observations classified as Class 1:", count_class_1)

KeyError: 'Prob_Class_1'